In [42]:
import os
os.getcwd()


'c:\\Users\\PC\\Documents\\DEF'

In [43]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('claim_data.csv')
df.head()

,EDI_CLAIM_ID,INTEG_MEMBER_NUMBER,DOB,SERVICE_TYPE,CENTRAL_ID,MEMBER_NUMBER,PATIENT_NAMES,BENEFIT_DESC,AMOUNT,PROV_INVOICE_NUMBER,TRANSACTION_DATE,DATE,PROVIDER_KEY,PROVIDER_NAME,CLAIM_TYPE,CLAIM_STATUS,CLAIM_HAS_ATTACHMENT,status 2,STATUS
0,30985874.0,2011993,17/12/2012,OP,386813357,DEFMIS-130360-02,MICHELLE KENDI,OUT PATIENT OVERALL,"5,096.32",PDRC260100020,01/01/2026,01/01/2026,SKSP_92,MATER HOSPITAL,Normal claim,Claim has been picked successfully,Claim has attachment,PAID,Paid
1,30986204.0,2002456,01/01/1960,OP,386813740,DEFMIS-12997-00,JAMES LOKELIMAN MAIYWA,OUT PATIENT OVERALL,"4,593.75",MN/488096/25,01/01/2026,01/01/2026,SKSP_489,CHERANGANY NURSING HOME,Normal claim,Claim has been picked successfully,Claim has attachment,PAID,Paid
2,30986230.0,2004698,01/01/1977,OP,386813679,DEFMIS-16286-01,AGNES KONO NJIKON,OUT PATIENT OVERALL,"3,135.00",NHL.54398,01/01/2026 09:41,01/01/2026,SKSP_5641,NAMO HOSPITAL - EQUITY STREET LODWAR,Normal claim,Claim has been picked successfully,Claim has attachment,PAID,Paid
3,30986294.0,2012235,31/12/1962,OP,468970558,DEFMIS-58940-00,ELMI ALI ALIFATA,OUT PATIENT OVERALL,"2,327.50",OP/240916760,01/01/2026,01/01/2026,SKSP_2063,MOTHER ANGELA HURUMA HOSPITAL,Normal claim,Claim has been picked successfully,Claim has attachment,PAID,Paid
4,30986326.0,2002782,28/06/1957,OP,386813931,DEFMIS-8980-00,PETER MWANGI GACHUHI,OUT PATIENT OVERALL,"18,720.00",IN01233,01/01/2026,01/01/2026,SKSP_7192,MONALIFE PHARMACEUTICAL LTD,Normal claim,Claim has been picked successfully,Claim has attachment,PAID,Paid


In [44]:
df.duplicated().sum()

0

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17153 entries, 0 to 17152
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   EDI_CLAIM_ID          17009 non-null  float64
 1   INTEG_MEMBER_NUMBER   17153 non-null  int64  
 2   DOB                   17153 non-null  object 
 3   SERVICE_TYPE          17153 non-null  object 
 4   CENTRAL_ID            17153 non-null  int64  
 5   MEMBER_NUMBER         17153 non-null  object 
 6   PATIENT_NAMES         17153 non-null  object 
 7   BENEFIT_DESC          17153 non-null  object 
 8   AMOUNT                17153 non-null  object 
 9   PROV_INVOICE_NUMBER   17124 non-null  object 
 10  TRANSACTION_DATE      17153 non-null  object 
 11  DATE                  17153 non-null  object 
 12  PROVIDER_KEY          17153 non-null  object 
 13  PROVIDER_NAME         17153 non-null  object 
 14  CLAIM_TYPE            17153 non-null  object 
 15  CLAIM_STATUS       

In [46]:
df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')

C:\Users\PC\AppData\Local\Temp\ipykernel_7684\441165960.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')


In [47]:
from datetime import datetime

today = pd.to_datetime("today")

df['AGE'] = (today - df['DOB']).dt.days // 365

In [48]:
bins = [0, 21, 40, 50, 60, 79, np.inf]
labels = [
    '0-21',
    '22-40',
    '41-50',
    '51-60',
    '61-79',
    '80+'
]

df['AGE_BRACKET'] = pd.cut(df['AGE'], bins=bins, labels=labels, right=True, include_lowest=True)

In [49]:
df.groupby('AGE_BRACKET')['INTEG_MEMBER_NUMBER'].nunique()

C:\Users\PC\AppData\Local\Temp\ipykernel_7684\398784925.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('AGE_BRACKET')['INTEG_MEMBER_NUMBER'].nunique()


AGE_BRACKET
0-21     1003
22-40     228
41-50     421
51-60    1334
61-79    2970
80+        29
Name: INTEG_MEMBER_NUMBER, dtype: int64

In [50]:
df['VISIT_KEY'] = (
    df['MEMBER_NUMBER'].astype(str) + '_' +
    df['DATE'].astype(str) + '_' +
    df['SERVICE_TYPE'].astype(str)
)

In [51]:
df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')

In [52]:
unique_visits = df.groupby('AGE_BRACKET')['VISIT_KEY'].nunique()
unique_visits

C:\Users\PC\AppData\Local\Temp\ipykernel_7684\3522360464.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_visits = df.groupby('AGE_BRACKET')['VISIT_KEY'].nunique()


AGE_BRACKET
0-21     1588
22-40     401
41-50     884
51-60    3059
61-79    8774
80+       138
Name: VISIT_KEY, dtype: int64

In [53]:
visit_summary = df.groupby('AGE_BRACKET').agg(
    unique_visits=('VISIT_KEY', 'nunique'),
    unique_members=('MEMBER_NUMBER', 'nunique'),
    total_claims=('EDI_CLAIM_ID', 'count'),
    total_amount=('AMOUNT', 'sum')
).reset_index()

visit_summary

C:\Users\PC\AppData\Local\Temp\ipykernel_7684\3327070554.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  visit_summary = df.groupby('AGE_BRACKET').agg(


,AGE_BRACKET,unique_visits,unique_members,total_claims,total_amount
0,0-21,1588,1004,1818,"5,096.328,185.031,108.003,278.102,100.0012,125..."
1,22-40,401,228,459,"2,565.008,475.001,467.001,600.002,850.00712.51..."
2,41-50,884,421,1052,"3,135.009,940.203,413.003,146.004,502.001,429...."
3,51-60,3059,1335,3444,"6,998.462,490.0016,548.003,533.003,300.005,861..."
4,61-79,8774,2970,10085,"4,593.752,327.5018,720.009,857.508,145.601,658..."
5,80+,138,29,151,"1,046.0024,569.908,138.0011,235.004,942.089,00..."


In [54]:
df['AMOUNT'] = (
    df['AMOUNT']
    .astype(str)
    .str.replace(',', '')
)

df['AMOUNT'] = pd.to_numeric(df['AMOUNT'], errors='coerce')

In [55]:
age_summary = df.groupby('AGE_BRACKET').agg(
    unique_visits=('VISIT_KEY', 'nunique'),
    unique_members=('MEMBER_NUMBER', 'nunique'),
    total_claims=('EDI_CLAIM_ID', 'count'),
    total_amount=('AMOUNT', 'sum')
).reset_index()

C:\Users\PC\AppData\Local\Temp\ipykernel_7684\2394139342.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_summary = df.groupby('AGE_BRACKET').agg(


In [56]:
age_summary

,AGE_BRACKET,unique_visits,unique_members,total_claims,total_amount
0,0-21,1588,1004,1818,1.848428e+07
1,22-40,401,228,459,4.905471e+06
2,41-50,884,421,1052,1.470895e+07
3,51-60,3059,1335,3444,4.689450e+07
4,61-79,8774,2970,10085,1.595933e+08
5,80+,138,29,151,4.764105e+06


In [57]:
age_summary['total_amount'] = age_summary['total_amount'].apply(lambda x: f"{x:,.0f}")
age_summary

,AGE_BRACKET,unique_visits,unique_members,total_claims,total_amount
0,0-21,1588,1004,1818,"18,484,277"
1,22-40,401,228,459,"4,905,471"
2,41-50,884,421,1052,"14,708,955"
3,51-60,3059,1335,3444,"46,894,503"
4,61-79,8774,2970,10085,"159,593,296"
5,80+,138,29,151,"4,764,105"


In [ ]:
benefit_summary = df.groupby('BENEFIT_DESC').agg(
    unique_visits=('VISIT_KEY', 'nunique'),
    total_amount=('AMOUNT', 'sum')
).reset_index()
benefit_summary['total_amount'] = benefit_summary['total_amount'].apply(lambda x: f"{x:,.0f}")
benefit_summary

In [62]:
benefit_summary = df.groupby('BENEFIT_DESC').agg(
    unique_visits=('VISIT_KEY', 'nunique'),
    total_amount=('AMOUNT', 'sum')
).reset_index()
benefit_summary['total_amount'] = benefit_summary['total_amount'].apply(lambda x: f"{x:,.2f}")
benefit_summary

,BENEFIT_DESC,unique_visits,total_amount
0,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,1105,"144,006,810.33"
1,OUT PATIENT DENTAL,311,"2,365,307.71"
2,OUT PATIENT OPTICAL,487,"5,126,621.93"
3,OUT PATIENT OVERALL,13094,"97,851,867.74"


In [63]:
df['MONTH'] = df['DATE'].dt.to_period('M')

In [65]:
monthly_cost = df.groupby(['MONTH', 'SERVICE_TYPE'])['AMOUNT'].sum().reset_index()
monthly_cost['MONTH'] = monthly_cost['MONTH'].dt.to_timestamp()
monthly_cost

,MONTH,SERVICE_TYPE,AMOUNT
0,2026-01-01,IP,947533.14
1,2026-01-01,OP,593110.96
2,2026-02-01,IP,3167906.39
3,2026-02-01,OP,4010397.95
4,2026-03-01,IP,5994533.09
5,2026-03-01,OP,3869651.37
6,2026-04-01,IP,5798058.10
7,2026-04-01,OP,2952354.74
8,2026-05-01,IP,3293752.20
9,2026-05-01,OP,4963784.00


In [70]:
monthly_pivot = monthly_cost.pivot(
    index='MONTH',
    columns='SERVICE_TYPE',
    values='AMOUNT'
).fillna(0)

monthly_pivot['TOTAL_COST'] = monthly_pivot.sum(axis=1)
monthly_pivot

SERVICE_TYPE,IP,OP,TOTAL_COST
MONTH,,,
2026-01-01,947533.14,593110.96,1540644.10
2026-02-01,3167906.39,4010397.95,7178304.34
2026-03-01,5994533.09,3869651.37,9864184.46
2026-04-01,5798058.10,2952354.74,8750412.84
2026-05-01,3293752.20,4963784.00,8257536.20
2026-06-01,5449365.95,3754909.84,9204275.79
2026-07-01,4140998.63,3082975.45,7223974.08
2026-08-01,3964154.80,2212597.77,6176752.57
2026-09-01,5316482.96,3933802.98,9250285.94
